In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [2]:
df_green = spark.read.parquet('data/pq/green/*/*')

In [3]:
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime') \
    .withColumnRenamed('RatecodeID', 'ratecode_id') \
    .withColumnRenamed('PULocationID', 'pickup_location_id') \
    .withColumnRenamed('DOLocationID', 'dropoff_location_id')

In [5]:
df_green.registerTempTable('green_taxi_data')

d:\Project\data-engineering-zoomcamp\.venv\Lib\site-packages\pyspark\sql\classic\dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [18]:
df_green_revenue = spark.sql("""
SELECT 
    -- Revenue grouping 
    date_trunc('hour', pickup_datetime) AS hour, 
    pickup_location_id AS zone,
    SUM(total_amount) AS total_amount,
    COUNT(*) AS number_records
FROM
    green_taxi_data
WHERE
    pickup_datetime > '2020-01-01 00:00:00'
GROUP BY
    1, 2
-- ORDER BY
--     1, 2
""")

In [16]:
df_green_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|      total_amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-01 00:00:00|   7|            769.73|            45|
|2020-01-01 00:00:00|  17|195.03000000000003|             9|
|2020-01-01 00:00:00|  18|               7.8|             1|
|2020-01-01 00:00:00|  22|              15.8|             1|
|2020-01-01 00:00:00|  24|              87.6|             3|
|2020-01-01 00:00:00|  25| 531.0000000000001|            26|
|2020-01-01 00:00:00|  29|              61.3|             1|
|2020-01-01 00:00:00|  32| 68.94999999999999|             2|
|2020-01-01 00:00:00|  33|            317.27|            11|
|2020-01-01 00:00:00|  35|            129.96|             5|
|2020-01-01 00:00:00|  36|295.34000000000003|            11|
|2020-01-01 00:00:00|  37|            175.67|             6|
|2020-01-01 00:00:00|  38| 98.78999999999999|             2|
|2020-01-01 00:00:00|  4

In [25]:
df_green_revenue.repartition(20).write.parquet('data/report/revenue/green', mode='overwrite') # coalesce to reduce partition

In [20]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')

In [21]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime') \
    .withColumnRenamed('RatecodeID', 'ratecode_id') \
    .withColumnRenamed('PULocationID', 'pickup_location_id') \
    .withColumnRenamed('DOLocationID', 'dropoff_location_id')

In [22]:
df_yellow.registerTempTable('yellow_taxi_data')

d:\Project\data-engineering-zoomcamp\.venv\Lib\site-packages\pyspark\sql\classic\dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [23]:
df_yellow_revenue = spark.sql("""
SELECT 
    -- Revenue grouping 
    date_trunc('hour', pickup_datetime) AS hour, 
    pickup_location_id AS zone,
    SUM(total_amount) AS total_amount,
    COUNT(*) AS number_records
FROM
    yellow_taxi_data
WHERE
    pickup_datetime > '2020-01-01 00:00:00'
GROUP BY
    1, 2
-- ORDER BY
--     1, 2
""")

In [26]:
df_yellow_revenue.repartition(20).write.parquet('data/report/revenue/yellow', mode='overwrite') # coalesce to reduce partition

In [33]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('total_amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

In [34]:
df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('total_amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [35]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour','zone'], how='outer')

In [36]:
df_join.show()

+-------------------+----+-----------------+--------------------+------------------+---------------------+
|               hour|zone|     green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+-----------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|  13|             NULL|                NULL|1214.8000000000002|                   56|
|2020-01-01 00:00:00|  18|              7.8|                   1|               5.8|                    1|
|2020-01-01 00:00:00|  83|94.09999999999998|                   7|               9.8|                    1|
|2020-01-01 00:00:00| 126|             NULL|                NULL|             170.6|                    2|
|2020-01-01 00:00:00| 134|            69.05|                   6|              NULL|                 NULL|
|2020-01-01 00:00:00| 161|             NULL|                NULL| 9410.209999999995|                  488|
|2020-01-01 00:00:00| 231|           

In [37]:
df_join.write.parquet('data/report/revenue/total')

In [2]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-08 00:29:53--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.172.0.48, 18.172.0.77, 18.172.0.12, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.172.0.48|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: 'taxi_zone_lookup.csv'

     0K .......... ..                                         100% 57.9M=0s

2026-03-08 00:29:53 (57.9 MB/s) - 'taxi_zone_lookup.csv' saved [12331/12331]



In [3]:
df_zone = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [4]:
df_zone.write.parquet('zones')

In [6]:
df_join = spark.read.parquet('data/report/revenue/total')

In [7]:
df_result = df_join.join(df_zone, df_join.zone == df_zone.LocationID)

In [8]:
df_result.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|LocationID|  Borough|                Zone|service_zone|
+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|2020-01-01 00:00:00|  45|              NULL|                NULL|            732.48|                   42|        45|Manhattan|           Chinatown| Yellow Zone|
|2020-01-01 00:00:00| 140|              NULL|                NULL|           2268.91|                  138|       140|Manhattan|     Lenox Hill East| Yellow Zone|
|2020-01-01 00:00:00| 196|             37.91|                   2|              35.6|                    2|       196|   Queens|           Rego Park|   Boro Zone|
|2020-01-01 00:00:00| 

In [10]:
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zone')